# Análise Comparativa de Representações

Comparação entre BoW, TF-IDF e embeddings no corpus de 40 manifestações.

## 1. Preparação dos dados

A categoria oficial é preservada sem correções. O objetivo é comparar representações, não alterar o rótulo administrativo.

In [ ]:

import json, re, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

with open("manifestacoes.json", encoding="utf-8") as f:
    dados = json.load(f)
df = pd.DataFrame(dados)
df["texto"] = df["texto"].fillna("")
textos = df["texto"].tolist()
ids = df["id"].tolist()
print(df.shape)
df.head()


## 2. BoW e TF-IDF

In [ ]:

bow = CountVectorizer(lowercase=True, strip_accents="unicode")
X_bow = bow.fit_transform(textos)
tfidf = TfidfVectorizer(lowercase=True, strip_accents="unicode")
X_tfidf = tfidf.fit_transform(textos)
print("BoW:", X_bow.shape, "TF-IDF:", X_tfidf.shape)


## 3. Similaridade dos pares exigidos

In [ ]:

pares = [("M003","M017"), ("M008","M022"), ("M008","M031")]
idx = {v:i for i,v in enumerate(ids)}
linhas=[]
for a,b in pares:
    i,j=idx[a],idx[b]
    linhas.append({
        "par": f"{a} × {b}",
        "BoW": cosine_similarity(X_bow[i], X_bow[j])[0,0],
        "TF-IDF": cosine_similarity(X_tfidf[i], X_tfidf[j])[0,0]
    })
comparacao = pd.DataFrame(linhas)
comparacao


## 4. Embeddings semânticos

O modelo multilíngue é carregado localmente pela `sentence-transformers`. Se houver internet na primeira execução, o modelo é baixado e depois fica em cache.

In [ ]:

from sentence_transformers import SentenceTransformer
MODELOS = [
    "paraphrase-multilingual-MiniLM-L12-v2",
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
]
modelo_nome = MODELOS[0]
modelo = SentenceTransformer(modelo_nome)
E = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=True)
for a,b in pares:
    i,j=idx[a],idx[b]
    print(a,b, float(np.dot(E[i],E[j])))
emb_rows=[]
for a,b in pares:
    i,j=idx[a],idx[b]
    emb_rows.append({"par":f"{a} × {b}","Embeddings":float(np.dot(E[i],E[j]))})
comparacao = comparacao.merge(pd.DataFrame(emb_rows), on="par")
comparacao


## 5. Visualização PCA dos embeddings

In [ ]:

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(E)
import matplotlib.pyplot as plt
plt.figure(figsize=(10,7))
for cat in df["categoria_oficial"].unique():
    m = df["categoria_oficial"].eq(cat)
    plt.scatter(coords[m,0], coords[m,1], label=cat)
    for k in np.where(m)[0]:
        plt.annotate(ids[k], (coords[k,0],coords[k,1]), fontsize=7)
plt.title("Espaço semântico — PCA dos embeddings")
plt.legend()
plt.show()


## 6. Conclusão

BoW e TF-IDF dependem fortemente de sobreposição lexical; por isso podem subestimar frases semanticamente equivalentes quando usam vocabulário diferente. Embeddings representam relações semânticas de forma mais adequada para busca e duplicatas, embora dependam do modelo escolhido, domínio e limiar. A tabela produzida acima deve ser usada como evidência quantitativa na discussão final.